# Notebook 3: Tournament Simulation & Bracket Optimization
## March Madness 2026 Projection System
---
**Key fix from v1:** Win probability engine now uses **team-level advanced stats**
(SRS, eFG%, ORtg, AdjOE) for each matchup — not just seed numbers.

**Input:** Trained models from `models/`, features from `data/features_2026.csv`
**Output:** `results/simulations.csv`, `results/optimal_bracket.csv`

In [1]:
# ============================================================
# 3.0 CONFIG & IMPORTS
# ============================================================
import os, json, pickle, warnings, logging, time
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("sim")

DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 51
SALT = 2026_03
SEED = RANDOM_SEED + SALT
rng = np.random.default_rng(SEED)

import multiprocessing
N_CORES = multiprocessing.cpu_count()

# ==========================================================
# ⚙️  SIMULATION CONFIG (pool-specific settings are in Notebook 5)
# ==========================================================
SIM_CONFIG = {
    "n_sims": 1_000_000,
    "batch_size": 25_000,
}

print(f"Simulation: {SIM_CONFIG['n_sims']:,} sims, batch={SIM_CONFIG['batch_size']:,}")

Simulation: 1,000,000 sims, batch=25,000


In [2]:
# ============================================================
# 3.1 LOAD MODELS, FEATURES & BUILD TEAM LOOKUP
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load scaler, imputer, feature columns
scaler = pickle.load(open(MODEL_DIR / "scaler.pkl", "rb"))
imputer = pickle.load(open(MODEL_DIR / "imputer.pkl", "rb"))
ALL_FEATURE_COLS = pickle.load(open(MODEL_DIR / "feature_cols.pkl", "rb"))

with open(MODEL_DIR / "ensemble_config.json") as f:
    ensemble_config = json.load(f)
ensemble_weights = ensemble_config["weights"]
log.info(f"Feature columns: {ALL_FEATURE_COLS}")
log.info(f"Ensemble weights: {ensemble_weights}")

# Load models
loaded_models = {}
for name, path in [("LogReg", "logistic_regression.pkl"), ("XGBoost", "xgboost.pkl"),
                    ("RandomForest", "random_forest.pkl")]:
    try:
        loaded_models[name] = pickle.load(open(MODEL_DIR / path, "rb"))
        log.info(f"  Loaded {name}")
    except: pass

try:
    loaded_models["BetaBinomial"] = pickle.load(open(MODEL_DIR / "beta_binomial.pkl", "rb"))
    log.info("  Loaded Beta-Binomial")
except: pass

try:
    import arviz as az
    loaded_models["BayesLR_trace"] = az.from_netcdf(str(MODEL_DIR / "bayesian_lr_trace.nc"))
    log.info("  Loaded Bayesian LR trace")
except: pass

# --- Load 2026 team features for matchup computation ---
features_2026 = pd.read_csv(DATA_DIR / "features_2026.csv")
bracket_2026 = pd.read_csv(DATA_DIR / "bracket_2026.csv")

# Build lookup: team_name → feature dict
team_features = {}
for _, row in features_2026.iterrows():
    team_features[row["team"]] = row.to_dict()

log.info(f"Loaded {len(loaded_models)} models, {len(team_features)} team feature sets")
log.info(f"Sample team features for Duke: {list(team_features.get('Duke', {}).keys())[:10]}...")

2026-03-19 13:01:14,763 [INFO] Feature columns: ['seed_diff', 'seed_sum', 'seed_diff_sq', 'abs_seed_diff', 'higher_seed_is_team1', 'round_x_seeddiff', 'is_upset_seed', 'seed1_sq', 'seed2_sq', 'seed_product', 'log_seed_ratio', 'SRS_diff', 'SOS_diff', 'eFG%_diff', 'TOV%_diff', 'ORtg_diff', 'Pace_diff', 'ORB%_diff', 'AdjOE_diff', 'AdjDE_diff', 'Barthag_diff', 'AdjTempo_diff', 'WAB_diff', 'Pace_avg', 'ORtg_avg', 'SRS_sum', 'AdjOE_avg', 'AdjDE_avg', 'AdjTempo_avg', 'OE_vs_DE_1', 'OE_vs_DE_2']
2026-03-19 13:01:14,764 [INFO] Ensemble weights: {'LogReg': 0.21106037590555668, 'XGBoost': 0.177611339176738, 'RandomForest': 0.20403442974046718, 'NeuralNet': 0.19814259902962553, 'BayesLR': 0.20915125614761274}
2026-03-19 13:01:14,764 [INFO]   Loaded LogReg
2026-03-19 13:01:14,794 [INFO]   Loaded XGBoost
2026-03-19 13:01:14,859 [INFO]   Loaded RandomForest
2026-03-19 13:01:14,859 [INFO]   Loaded Beta-Binomial
2026-03-19 13:01:15,043 [INFO] arviz_base not installed
2026-03-19 13:01:15,043 [INFO] arvi

In [3]:
# ============================================================
# 3.2 WIN PROBABILITY ENGINE (WITH ADVANCED STATS)
# ============================================================
def compute_matchup_features(team1_name, team2_name, round_num=1):
    """
    Build the full feature vector for a matchup using team-level stats.
    Includes seed features, differential stats, and level/environment features.
    """
    t1 = team_features.get(team1_name, {})
    t2 = team_features.get(team2_name, {})

    seed1 = t1.get("seed", 8)
    seed2 = t2.get("seed", 8)
    seed_diff = seed1 - seed2

    features = {}

    # --- Seed features ---
    features["seed_diff"] = seed_diff
    features["seed_sum"] = seed1 + seed2
    features["seed_diff_sq"] = seed_diff ** 2
    features["abs_seed_diff"] = abs(seed_diff)
    features["higher_seed_is_team1"] = int(seed1 < seed2)
    features["round_x_seeddiff"] = round_num * seed_diff
    features["is_upset_seed"] = int(seed1 > seed2)
    features["seed1_sq"] = seed1 ** 2
    features["seed2_sq"] = seed2 ** 2
    features["seed_product"] = seed1 * seed2
    features["log_seed_ratio"] = np.log(max(seed2, 0.5) / max(seed1, 0.5))

    # --- Differential stat features (team1 - team2) ---
    diff_pairs = {
        "SRS_diff": ("SRS", "SRS"),
        "SOS_diff": ("SOS", "SOS"),
        "eFG%_diff": ("eFG%", "eFG%"),
        "TOV%_diff": ("TOV%", "TOV%"),
        "ORtg_diff": ("ORtg", "ORtg"),
        "Pace_diff": ("Pace", "Pace"),
        "ORB%_diff": ("ORB%", "ORB%"),
        "AdjOE_diff": ("bt_AdjOE", "bt_AdjOE"),
        "AdjDE_diff": ("bt_AdjDE", "bt_AdjDE"),
        "Barthag_diff": ("bt_Barthag", "bt_Barthag"),
        "AdjTempo_diff": ("bt_AdjTempo", "bt_AdjTempo"),
        "WAB_diff": ("bt_WAB", "bt_WAB"),
    }
    for feat_name, (col1, col2) in diff_pairs.items():
        if feat_name in ALL_FEATURE_COLS:
            v1 = t1.get(col1, np.nan)
            v2 = t2.get(col2, np.nan)
            try:
                features[feat_name] = float(v1) - float(v2)
            except (TypeError, ValueError):
                features[feat_name] = np.nan

    # --- Level features (absolute game environment) ---
    level_pairs = {
        "Pace_avg": ("Pace", "Pace"),
        "ORtg_avg": ("ORtg", "ORtg"),
        "SRS_sum": ("SRS", "SRS"),
        "AdjOE_avg": ("bt_AdjOE", "bt_AdjOE"),
        "AdjDE_avg": ("bt_AdjDE", "bt_AdjDE"),
        "AdjTempo_avg": ("bt_AdjTempo", "bt_AdjTempo"),
    }
    for feat_name, (col1, col2) in level_pairs.items():
        if feat_name in ALL_FEATURE_COLS:
            v1 = t1.get(col1, np.nan)
            v2 = t2.get(col2, np.nan)
            try:
                if "sum" in feat_name:
                    features[feat_name] = float(v1) + float(v2)
                else:
                    features[feat_name] = (float(v1) + float(v2)) / 2
            except (TypeError, ValueError):
                features[feat_name] = np.nan

    # Cross terms: each team's offense vs opponent's defense
    if "OE_vs_DE_1" in ALL_FEATURE_COLS:
        try:
            features["OE_vs_DE_1"] = float(t1.get("bt_AdjOE", np.nan)) - float(t2.get("bt_AdjDE", np.nan))
            features["OE_vs_DE_2"] = float(t2.get("bt_AdjOE", np.nan)) - float(t1.get("bt_AdjDE", np.nan))
        except (TypeError, ValueError):
            features["OE_vs_DE_1"] = np.nan
            features["OE_vs_DE_2"] = np.nan

    # Build array in correct column order
    feat_array = np.array([[features.get(c, np.nan) for c in ALL_FEATURE_COLS]])
    return feat_array


def get_win_probability(team1_name, team2_name, round_num=1, use_posterior=False, rng=None):
    """
    Ensemble win probability for team1 vs team2 using team-level features.
    """
    raw_features = compute_matchup_features(team1_name, team2_name, round_num)
    features_imputed = imputer.transform(raw_features)
    features_scaled = scaler.transform(features_imputed)

    probs = {}

    # Frequentist models
    for name in ["LogReg", "XGBoost", "RandomForest"]:
        if name in loaded_models:
            probs[name] = loaded_models[name].predict_proba(features_scaled)[0, 1]

    # Beta-Binomial
    if "BetaBinomial" in loaded_models:
        bb = loaded_models["BetaBinomial"]
        t1 = team_features.get(team1_name, {})
        t2 = team_features.get(team2_name, {})
        s1, s2 = int(t1.get("seed", 8)), int(t2.get("seed", 8))
        key = (min(s1, s2), max(s1, s2))
        if key in bb:
            params = bb[key]
            if use_posterior and rng is not None:
                p_lower = rng.beta(params["alpha"], params["beta"])
            else:
                p_lower = params["mean"]
            probs["BetaBinomial"] = p_lower if s1 <= s2 else 1 - p_lower
        else:
            probs["BetaBinomial"] = 1 / (1 + 10 ** ((s1 - s2) / 8))

    # Bayesian LR
    if "BayesLR_trace" in loaded_models:
        trace = loaded_models["BayesLR_trace"]
        if use_posterior and rng is not None:
            chain = rng.integers(trace.posterior["betas"].shape[0])
            draw = rng.integers(trace.posterior["betas"].shape[1])
            beta_sample = trace.posterior["betas"].values[chain, draw, :]
            alpha_sample = trace.posterior["alpha"].values[chain, draw]
        else:
            beta_sample = trace.posterior["betas"].values.mean(axis=(0, 1))
            alpha_sample = trace.posterior["alpha"].values.mean()
        logit = alpha_sample + features_scaled[0] @ beta_sample
        probs["BayesLR"] = 1 / (1 + np.exp(-logit))

    if not probs:
        t1 = team_features.get(team1_name, {})
        t2 = team_features.get(team2_name, {})
        s1, s2 = t1.get("seed", 8), t2.get("seed", 8)
        return 1 / (1 + 10 ** ((s1 - s2) / 8))

    total_w = 0
    ensemble_p = 0
    for name, p in probs.items():
        w = ensemble_weights.get(name, 0.05)
        ensemble_p += w * np.clip(p, 0.01, 0.99)
        total_w += w

    return ensemble_p / max(total_w, 1e-8)


# --- Quick test ---
test_matchups = [("Duke", "Siena"), ("Ohio St.", "TCU"), ("St. John's", "Northern Iowa"),
                 ("Duke", "Michigan"), ("Furman", "UConn")]
for t1, t2 in test_matchups:
    p = get_win_probability(t1, t2)
    s1 = team_features.get(t1, {}).get("seed", "?")
    s2 = team_features.get(t2, {}).get("seed", "?")
    log.info(f"  ({s1}){t1} vs ({s2}){t2}: P(team1)={p:.4f}")

2026-03-19 13:01:15,998 [INFO]   (1)Duke vs (16)Siena: P(team1)=0.9755
2026-03-19 13:01:16,038 [INFO]   (8)Ohio St. vs (9)TCU: P(team1)=0.8086
2026-03-19 13:01:16,077 [INFO]   (5)St. John's vs (12)Northern Iowa: P(team1)=0.8312
2026-03-19 13:01:16,117 [INFO]   (1)Duke vs (1)Michigan: P(team1)=0.4157
2026-03-19 13:01:16,156 [INFO]   (15)Furman vs (2)UConn: P(team1)=0.0849


In [4]:
# ============================================================
# 3.3 BRACKET SIMULATION ENGINE
# ============================================================
def simulate_tournament(bracket_df, n_sims, rng, use_posterior=True):
    """Simulate full tournament n_sims times using team-level features."""
    t0 = time.time()

    # Organize bracket: standard NCAA bracket order per region
    bracket_order = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
    region_order = ["East", "West", "South", "Midwest"]

    team_names = []
    team_seeds = []
    team_regions = []

    for region in region_order:
        reg = bracket_df[bracket_df["region"] == region]
        for seed in bracket_order:
            row = reg[reg["seed"] == seed]
            if len(row) > 0:
                team_names.append(row.iloc[0]["team"])
                team_seeds.append(seed)
                team_regions.append(region)
            else:
                team_names.append(f"MISSING_{region}_{seed}")
                team_seeds.append(seed)
                team_regions.append(region)

    n_teams = len(team_names)
    team_seeds = np.array(team_seeds)

    # Pre-compute win probs for all matchups at each round
    # round_x_seeddiff feature means probabilities shift by round
    log.info("Pre-computing matchup probabilities (per round)...")
    prob_cache = {}  # key = (team_i, team_j, round_num)
    for rnd in range(1, 7):
        for i in range(n_teams):
            for j in range(i+1, n_teams):
                p = get_win_probability(team_names[i], team_names[j], round_num=rnd)
                prob_cache[(i, j, rnd)] = p
                prob_cache[(j, i, rnd)] = 1 - p

    log.info(f"Cached {len(prob_cache)} matchup probabilities (6 rounds)")
    log.info(f"Simulating {n_sims:,} tournaments...")

    advancement_counts = np.zeros((n_teams, 7), dtype=np.int32)
    all_champions = []
    batch_size = SIM_CONFIG["batch_size"]
    n_batches = (n_sims + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        actual_batch = min(batch_size, n_sims - batch_idx * batch_size)
        current = np.tile(np.arange(n_teams), (actual_batch, 1))

        for rnd_idx in range(6):
            n_current = current.shape[1]
            n_games = n_current // 2

            for sim_idx in range(actual_batch):
                for team_idx in current[sim_idx]:
                    advancement_counts[team_idx, rnd_idx] += 1

            winners = np.zeros((actual_batch, n_games), dtype=np.int32)
            for g in range(n_games):
                t1_indices = current[:, g * 2]
                t2_indices = current[:, g * 2 + 1]

                probs_arr = np.array([
                    prob_cache.get((t1_indices[i], t2_indices[i], rnd_idx + 1), 0.5)
                    for i in range(actual_batch)
                ])

                if use_posterior:
                    noise = rng.normal(0, 0.02, size=actual_batch)
                    probs_arr = np.clip(probs_arr + noise, 0.01, 0.99)

                outcomes = rng.random(actual_batch) < probs_arr
                winners[:, g] = np.where(outcomes, t1_indices, t2_indices)

            current = winners

        for sim_idx in range(actual_batch):
            champ = current[sim_idx, 0]
            advancement_counts[champ, 6] += 1
            all_champions.append(champ)

        if (batch_idx + 1) % 5 == 0 or batch_idx == 0:
            elapsed = time.time() - t0
            pct = (batch_idx + 1) / n_batches * 100
            log.info(f"  Batch {batch_idx+1}/{n_batches} ({pct:.0f}%) - {elapsed:.1f}s")

    elapsed = time.time() - t0
    log.info(f"Done: {n_sims:,} brackets in {elapsed:.1f}s ({n_sims/elapsed:,.0f}/sec)")

    round_names = ["R64", "R32", "S16", "E8", "F4", "Championship", "Champion"]
    results = []
    for i in range(n_teams):
        row = {"team": team_names[i], "seed": team_seeds[i], "region": team_regions[i]}
        for j, rnd in enumerate(round_names):
            row[f"prob_{rnd}"] = advancement_counts[i, j] / n_sims
        results.append(row)

    return pd.DataFrame(results).sort_values("prob_Champion", ascending=False), np.array(all_champions)

sim_results, all_champs = simulate_tournament(bracket_2026, SIM_CONFIG["n_sims"], rng)

print("\n Championship Probabilities (Top 16):")
display(sim_results[["team", "seed", "region", "prob_F4", "prob_Championship", "prob_Champion"]].head(16))

2026-03-19 13:01:16,168 [INFO] Pre-computing matchup probabilities (per round)...
2026-03-19 13:09:02,742 [INFO] Cached 24192 matchup probabilities (6 rounds)
2026-03-19 13:09:02,742 [INFO] Simulating 1,000,000 tournaments...
2026-03-19 13:09:03,444 [INFO]   Batch 1/40 (2%) - 467.3s
2026-03-19 13:09:06,266 [INFO]   Batch 5/40 (12%) - 470.1s
2026-03-19 13:09:09,756 [INFO]   Batch 10/40 (25%) - 473.6s
2026-03-19 13:09:13,258 [INFO]   Batch 15/40 (38%) - 477.1s
2026-03-19 13:09:16,777 [INFO]   Batch 20/40 (50%) - 480.6s
2026-03-19 13:09:20,280 [INFO]   Batch 25/40 (62%) - 484.1s
2026-03-19 13:09:23,781 [INFO]   Batch 30/40 (75%) - 487.6s
2026-03-19 13:09:27,304 [INFO]   Batch 35/40 (88%) - 491.1s
2026-03-19 13:09:30,816 [INFO]   Batch 40/40 (100%) - 494.7s
2026-03-19 13:09:30,816 [INFO] Done: 1,000,000 brackets in 494.7s (2,022/sec)



 Championship Probabilities (Top 16):


,team,seed,region,prob_F4,prob_Championship,prob_Champion
48,Michigan,1,Midwest,0.599039,0.396331,0.254210
0,Duke,1,East,0.647285,0.381520,0.215579
16,Arizona,1,West,0.549849,0.340313,0.171177
32,Florida,1,South,0.382834,0.174705,0.091167
30,Purdue,2,West,0.270419,0.119947,0.064874
46,Houston,2,South,0.251313,0.122200,0.057285
42,Illinois,3,South,0.225206,0.125768,0.047606
62,Iowa St.,2,Midwest,0.178174,0.076167,0.028422
14,UConn,2,East,0.085453,0.030288,0.009322
10,Michigan St.,3,East,0.082510,0.029179,0.008878


In [5]:
# ============================================================
# 3.4 SAVE RESULTS
# ============================================================
sim_results.to_csv(RESULTS_DIR / "simulation_results.csv", index=False)

champ_counts = pd.Series(all_champs).value_counts()
champ_dist = pd.DataFrame({"team_idx": champ_counts.index, "count": champ_counts.values,
                            "probability": champ_counts.values / len(all_champs)})
champ_dist.to_csv(RESULTS_DIR / "champion_distribution.csv", index=False)

run_config = {
    "sim_config": SIM_CONFIG, "seed": SEED, "n_sims_actual": len(all_champs),
    "timestamp": datetime.now().isoformat(), "models_used": list(ensemble_weights.keys()),
}
with open(RESULTS_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2, default=str)

log.info(f"Results saved to {RESULTS_DIR}/")

print("\n Championship Probabilities (Top 16):")
cols = ["team", "seed", "region", "prob_F4", "prob_Championship", "prob_Champion"]
display(sim_results[[c for c in cols if c in sim_results.columns]].head(16))

print("\n Simulation complete! Run Notebooks 4, 5, 6.")

2026-03-19 13:09:30,862 [INFO] Results saved to results/



 Championship Probabilities (Top 16):


,team,seed,region,prob_F4,prob_Championship,prob_Champion
48,Michigan,1,Midwest,0.599039,0.396331,0.254210
0,Duke,1,East,0.647285,0.381520,0.215579
16,Arizona,1,West,0.549849,0.340313,0.171177
32,Florida,1,South,0.382834,0.174705,0.091167
30,Purdue,2,West,0.270419,0.119947,0.064874
46,Houston,2,South,0.251313,0.122200,0.057285
42,Illinois,3,South,0.225206,0.125768,0.047606
62,Iowa St.,2,Midwest,0.178174,0.076167,0.028422
14,UConn,2,East,0.085453,0.030288,0.009322
10,Michigan St.,3,East,0.082510,0.029179,0.008878



 Simulation complete! Run Notebooks 4, 5, 6.


## Simulation Summary

**Key improvement:** Win probabilities now incorporate team-level SRS, eFG%, ORtg, Pace — not just seed numbers. Two teams with the same seed but different SRS ratings will get different win probabilities.

**Next:** Open `04_evaluation.ipynb`